<a href="https://colab.research.google.com/github/bettywang711/eng-intern-assessment-data/blob/main/ir_assignment_Q1_Q4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: IR

## Preparations
* Put all your imports, and path constants in the next cells

In [1]:
!pip install whoosh
!pip install pytrec_eval
!pip install wget
!pip install chromadb
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp313-cp313-linux_x86_64.whl size=316472 sha256=158237b2e9b7326acddf595a0e941829ba31a7d7741473be53108d487e3f835c
  Stored in directory: /root/.cache/pip/wheels/2b/ee/4a/bee1cc81bea848461859dc7c6e9ae753cdcbaef8ab65eba4f6
Successfully built pytrec_eval
  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=5c3e2c164d1b39b574587af542e3aef5a5833ca21339b22617bbccaa11efe732
  Stored in directory: /root/.cache/pip/wheels/8a/b8/04/0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import wget
wget.download("https://github.com/MIE451-1513-2026/course-datasets/raw/main/product_search.zip", "product_search.zip")

'product_search.zip'

In [3]:
!unzip /content/product_search.zip

Streaming output truncated to the last 5000 lines.
  inflating: product_search/documents/1418253  
  inflating: product_search/documents/709533  
  inflating: product_search/documents/1600305  
  inflating: product_search/documents/504385  
  inflating: product_search/documents/1483746  
  inflating: product_search/documents/1622869  
  inflating: product_search/documents/3843  
  inflating: product_search/documents/1566793  
  inflating: product_search/documents/1095046  
  inflating: product_search/documents/1555783  
  inflating: product_search/documents/452813  
  inflating: product_search/documents/376069  
  inflating: product_search/documents/1036399  
  inflating: product_search/documents/863552  
  inflating: product_search/documents/1232032  
  inflating: product_search/documents/973474  
  inflating: product_search/documents/1084796  
  inflating: product_search/documents/965949  
  inflating: product_search/documents/722383  
  inflating: product_search/documents/1564202  


In [4]:
# imports DO NOT MODIFY
from whoosh import index, writing
from whoosh.searching import Results
from whoosh.fields import Schema, TEXT, KEYWORD, ID, STORED
from whoosh.analysis import *
from whoosh.qparser import QueryParser
from whoosh import qparser, query
from whoosh import scoring
import os.path
from pathlib import Path
import tempfile
import subprocess
import pytrec_eval
import wget
import abc
from abc import abstractmethod
from whoosh.analysis import Filter
import numpy as np
import chromadb
import json
from sentence_transformers import SentenceTransformer
import nltk
from nltk.stem import *
nltk.download("wordnet")

# Put all your imports here

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [5]:
class IRSystem(metaclass=abc.ABCMeta):
    """
    Abstract class which is inherited by other IR system

    DO NOT ADD OR MODIFY THIS CLASS
    """

    def __init__(self, data_dir):
        # DON'T change the following names,topic_file, qrels_file, document_dir, file_list, text_embedding
        self.topic_file = os.path.join(data_dir, "product.topics")
        self.qrels_file = os.path.join(data_dir, "product.qrels")
        self.document_dir = os.path.join(data_dir, "documents")
        self.file_list = [str(filePath) for filePath in Path(self.document_dir).glob("**/*") if filePath.is_file()]
        # this is the text embedding of the documents using Qwen/Qwen3-embedding-0.6b
        self.text_embedding = json.load(open(os.path.join(data_dir, "corpus_embeddings.json"), "r"))

        self.create_index()
        self.create_parser_searcher()

    @abstractmethod
    def create_index(self):
        pass

    @abstractmethod
    def add_files(self):
        pass

    @abstractmethod
    def create_parser_searcher(self):
        pass

    @abstractmethod
    def perform_search(self, topic_phrase):
        pass

    @staticmethod
    def post_process_score(score):
        return score

    @staticmethod
    def print_trec_eval_result(results):

        if not results:
            print('empty results')
            return

        def print_line(name, scope, num):
            print('{:25s}{:8s}{:.4f}'.format(name, scope, num))

        for query_id, query_measures in results.items():
            for measure, value in query_measures.items():
                if measure == "runid":
                    continue
                print_line(measure, query_id, value)

        for measure in query_measures.keys():
            if measure == "runid":
                continue
            print_line(
                measure,
                'all',
                pytrec_eval.compute_aggregated_measure(
                    measure,
                    [query_measures[measure]
                     for query_measures in results.values()]))

    def print_rel_name(self, q_id):
        with open(self.topic_file, "r") as tf:
            topics = tf.read().splitlines()
        for topic in topics:
            topic_id, topic_phrase = tuple(topic.split(" ", 1))
            if topic_id == q_id:
                print("---------------------------Topic_id and Topic_phrase----------------------------------")
                print(topic_id, topic_phrase)
                 # get search result
                topicResults = self.perform_search(topic_phrase)
                print("---------------------------Return documents----------------------------------")
                if isinstance(topicResults, dict):
                    # format the result for chroma search
                    for (docnum, result) in enumerate(topicResults['ids'][0]):
                        score = topicResults['distances'][0][docnum]
                        score = self.post_process_score(score)
                        print("%s Q0 %s %d %lf test" % (topic_id, os.path.basename(result), docnum, score))
                else:
                    # format the result for whoosh search
                    for (docnum, result) in enumerate(topicResults):
                        score = topicResults.score(docnum)
                        score = self.post_process_score(score)
                        print("%s Q0 %s %d %lf test" % (topic_id, os.path.basename(result["file_path"]), docnum, score))
                print("---------------------------Relevant documents----------------------------------")
                with open(self.qrels_file, 'r') as f_qrel:
                    qrels = f_qrel.readlines()
                    for i in qrels:
                        qid, _, doc, rel = i.rstrip().split(" ")
                        if qid == q_id and rel == "1":
                            print(i.rstrip())

    def py_trec_eval(self):

        self.create_parser_searcher()
        # Load topic file - a list of topics(search phrases) used for evalutation
        with open(self.topic_file, "r") as tf:
            topics = tf.read().splitlines()

            # create an output file to which we'll write our results
        temp_output_file = tempfile.mkstemp()[1]
        with open(temp_output_file, "w") as outputTRECFile:
            # for each evaluated topic:
            # build a query and record the results in the file in TREC_EVAL format
            for topic in topics:
                topic_id, topic_phrase = tuple(topic.split(" ", 1))
                # get search result
                topicResults = self.perform_search(topic_phrase)
                if isinstance(topicResults, dict):
                    # format the result for chroma search
                    for (docnum, result) in enumerate(topicResults['ids'][0]):
                        score = -topicResults['distances'][0][docnum]
                        outputTRECFile.write("%s Q0 %s %d %lf test\n" % (topic_id,
                        os.path.basename(result), docnum, score))
                else:
                    # format the result for whoosh search
                    for (docnum, result) in enumerate(topicResults):
                        score = topicResults.score(docnum)
                        outputTRECFile.write(
                            "%s Q0 %s %d %lf test\n" % (topic_id, os.path.basename(result["file_path"]), docnum, score))

        with open(self.qrels_file, 'r') as f_qrel:
            qrel = pytrec_eval.parse_qrel(f_qrel)

        with open(temp_output_file, 'r') as f_run:
            run = pytrec_eval.parse_run(f_run)

        evaluator = pytrec_eval.RelevanceEvaluator(
            qrel, pytrec_eval.supported_measures)

        results = evaluator.evaluate(run)

        self.print_trec_eval_result(results)


In [6]:
# Dont change this! Use it as-is in your code
# This filter will run for both the index and the query
class CustomFilter(Filter):
    is_morph = True
    def __init__(self, filterFunc, *args, **kwargs):
        self.customFilter = filterFunc
        self.args = args
        self.kwargs = kwargs
    def __eq__(self):
        return (other
                and self.__class__ is other.__class__)
    def __call__(self, tokens):
        for t in tokens:
            if t.mode == 'query': # if called by query parser
                t.text = self.customFilter(t.text, *self.args, **self.kwargs)
                yield t
            else: # == 'index' if called by indexer
                t.text = self.customFilter(t.text, *self.args, **self.kwargs)
                yield t

## Question 1


### Q1 (a): Provide answer to Q1 (a) here [markdown cell]

P10 (Precision at 10)

### Q1 (b): Provide answer to Q1 (b) here [markdown cell]

Precision at 10 is appropriate for product search because users usually focus on the first page of results. P10 measures the proportion of the top 10 retrieved products that are relevant, so it directly evaluates how useful the most visible search results are to the user.


## Question 2

### Q2 (a): Write your code below

**1. The auto-grader will extract and use the following variables, DON'T change the their names:**

      self.topic_file  
      self.qrels_file  
      self.document_dir   
      self.file_list  
      self.index_sys  
      self.query_parser  
      self.searcher   



**2. DON'T change the names of the already defined funtions**  
**3. DON'T change the py_trec_eval function**  
**4. DON'T change the class names including CustomFilter, IRSystem, IRQ2, IRQ3, IRQ4**  
**5. DON'T change the CustomFilter class and DON'T create any new custom filter class that is used to define Whoosh schema**

AI assistance disclosure: I used ChatGPT to help clarify parts of code I was unsure of and for explaining IRQ2, IRQ3, and IRQ4. However, I reviewed, ran, and evaluated the code and can explain the submitted solutions.


In [7]:
class IRQ2(IRSystem):
    def create_index(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Please update self.index_sys which should have type whoosh.index.FileIndex
        """

        # DON't change the name of 'index_sys'

        #Define the fields stored and searched in each indexed document
        schema = Schema (
            file_path = ID(stored=True),
            file_content = TEXT(analyzer=RegexTokenizer())
        )

        # Create a temporary directory and an empty Woosh index
        index_dir = tempfile.mkdtemp()
        self.index_sys = index.create_in(index_dir, schema)

    def add_files(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Add buffer to self.index_sys
        """
        # Open a buffered writer to efficiently add documents to the index
        writer = writing.BufferedWriter(
            self.index_sys,
            period=None,
            limit=1000
        )

        # Read every product file and add its path and content to the index
        try:
          for file_path in self.file_list:
            with open(file_path, encoding="utf-8") as file:
              file_content = file.read()
              writer.add_document(
                    file_path=file_path,
                    file_content=file_content
                    )
        finally:
        # Close the index and commit the index documents
          writer.close()

    def create_parser_searcher(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Please update self.query_parser and self.self.searcherwhich should have type whoosh.qparser.default.QueryParser and whoosh.searching.Searcher respectively
        """
        # DON't change the names of 'query_parser' and 'searcher'

        # Create a parser that searches the file_content field
        self.query_parser = QueryParser (
            "file_content",
            schema = self.index_sys.schema)

        # Create the object that runs queries against the index
        self.searcher = self.index_sys.searcher()

    def perform_search(self, topic_phrase):
        """
        INPUT:
            topic_phrase: string
        OUTPUT:
            topicResults: whoosh.searching.Results

        NOTE: Utilize self.query_parser and self.searcher to calculate the result for topic_phrase
        """
        #  Convert the text phrase into a Whoosh query object
        query = self.query_parser.parse(topic_phrase)

        # Search the index and return all matching documents
        topicResults = self.searcher.search(query, limit = None)
        return topicResults

In [8]:
q2 = IRQ2("product_search")

In [9]:
q2.add_files()

In [10]:
q2.py_trec_eval()

num_q                    100246  1.0000
num_ret                  100246  2.0000
num_rel                  100246  102.0000
num_rel_ret              100246  2.0000
map                      100246  0.0196
gm_map                   100246  -3.9318
Rprec                    100246  0.0196
bpref                    100246  0.0196
recip_rank               100246  1.0000
iprec_at_recall_0.00     100246  1.0000
iprec_at_recall_0.10     100246  0.0000
iprec_at_recall_0.20     100246  0.0000
iprec_at_recall_0.30     100246  0.0000
iprec_at_recall_0.40     100246  0.0000
iprec_at_recall_0.50     100246  0.0000
iprec_at_recall_0.60     100246  0.0000
iprec_at_recall_0.70     100246  0.0000
iprec_at_recall_0.80     100246  0.0000
iprec_at_recall_0.90     100246  0.0000
iprec_at_recall_1.00     100246  0.0000
P_5                      100246  0.4000
P_10                     100246  0.2000
P_15                     100246  0.1333
P_20                     100246  0.1000
P_30                     100246  0.06

In [11]:
q2.print_rel_name('100246')

---------------------------Topic_id and Topic_phrase----------------------------------
100246 SPLOTY Tire Inflator Air Compressor Portable 12V
---------------------------Return documents----------------------------------
100246 Q0 1555783 0 54.753307 test
100246 Q0 1489405 1 54.349004 test
---------------------------Relevant documents----------------------------------
100246 0 10037 1
100246 0 1030471 1
100246 0 103372 1
100246 0 1044 1
100246 0 1044436 1
100246 0 106086 1
100246 0 1079438 1
100246 0 1086239 1
100246 0 1093412 1
100246 0 10966 1
100246 0 1105375 1
100246 0 1107134 1
100246 0 1137791 1
100246 0 1140524 1
100246 0 1148601 1
100246 0 116655 1
100246 0 1176585 1
100246 0 117671 1
100246 0 1187066 1
100246 0 1191987 1
100246 0 120310 1
100246 0 1214125 1
100246 0 1214399 1
100246 0 121627 1
100246 0 1216577 1
100246 0 1267443 1
100246 0 128561 1
100246 0 1294856 1
100246 0 1306587 1
100246 0 1307993 1
100246 0 1349887 1
100246 0 1384746 1
100246 0 138718 1
100246 0 1418253 

In [33]:
INDEX_Q2 = q2.index_sys
QP_Q2 = q2.query_parser
SEARCHER_Q2 = q2.searcher

### Q2 (b): Provide answer to Q2 (b) here [markdown cell]

The baseline Whoosh system achieved an overall P_10 score of 0.2800.

### Q2 (c): Provide answer to Q2(c) here [markdown cell]

The P_10 results were:

- 100246: 0.2000
- 100278: 0.1000
- 100350: 0.7000
- 100436: 0.0000
- 200013: 1.0000
- 200206: 0.1000
- 200255: 0.4000
- 200292: 0.1000
- 200375: 0.1000
- 200377: 0.1000

Topic IDs 200013 and 100350 performed particularly well, with P@10 scores of 1.0 and 0.7 respectively. Topic ID 100436 performed particularly badly with a P@10 score of 0, while topic IDs 100278, 200206, and 200292 also performed poorly with scores of 0.1.


## Question 3

### Q3 (a): Provide answer to Q3 (a) here [markdown cell]

I picked topic ID 100350 and the query is "Google Wi-Fi System Mesh".

Document 1453918 is a false positive because the document appeared in the Return documents list but not the relevant document list. It is a  smartwatch rather than a home Wi-Fi mesh system. However, its description  contains all the query terms: "Google" appears in Google Pay, "Wi-Fi" describes its connectivity, "system" appears in "Operating System", and "mesh" describes its watchband. Whoosh matches these individual terms without understanding how they are related, causing the irrelevant product to be highly ranked.

Contrastingly, Document 1050182 is a false negative because it appears under relevant documents but did not appear under return documents. It is a Google Wi-Fi system, so it is directly relevant to the query. However, its description does not contain the exact word "mesh". Since Whoosh depends on exact term matching, it fails to retrieve the relevant product.

These errors suggest that the baseline Whoosh system could be improved by normalizing terms, permitting partial query matching, and giving greater weight to query terms that appear close together.


In [12]:
q2.print_rel_name("100350")

---------------------------Topic_id and Topic_phrase----------------------------------
100350 Google Wi-Fi System Mesh
---------------------------Return documents----------------------------------
100350 Q0 1057723 0 37.474892 test
100350 Q0 1362417 1 34.173613 test
100350 Q0 266325 2 32.038937 test
100350 Q0 1294680 3 31.956234 test
100350 Q0 1564873 4 30.456599 test
100350 Q0 917542 5 30.456599 test
100350 Q0 1453918 6 30.006657 test
100350 Q0 1252683 7 29.956002 test
100350 Q0 261846 8 29.867939 test
100350 Q0 408461 9 29.305432 test
100350 Q0 267392 10 28.957868 test
100350 Q0 830677 11 28.785175 test
100350 Q0 111297 12 28.128306 test
100350 Q0 1502886 13 26.953731 test
100350 Q0 1520654 14 24.817341 test
100350 Q0 1592144 15 21.839335 test
---------------------------Relevant documents----------------------------------
100350 0 1050182 1
100350 0 1057723 1
100350 0 1110923 1
100350 0 1294680 1
100350 0 1362417 1
100350 0 1425694 1
100350 0 1504399 1
100350 0 1532007 1
100350 0 156

In [13]:
document_ids = ["1453918", "1050182"]

for document_id in document_ids:
    file_path = os.path.join(q2.document_dir, document_id)

    with open(file_path, "r", encoding="utf-8") as file:
        print("Document ID:", document_id)
        print(file.read())
        print("----------------------------------------")

Document ID: 1453918
Skagen Connected Falster 2 Stainless Steel Magnetic Mesh Touchscreen Smartwatch, Color: Rose Gold (Model: SKT5103)
Product Description "Operating System: Wear OS by Google Sensors: Optical Heart Rate, Ambient Light, Barometric Altimeter, GPS, Accelerometer, Gyroscope, Microphone Processor: Qualcomm Snapdragon Wear 2100 Connectivity: Bluetooth(R) 4.1 Low Energy and Wi-Fi 802.11 b/g/n Storage: 4GB Water Resistant: Swimproof up to 30M, passes 10K stroke test What's in the Box: Smartwatch, USB Charging Cable, Quick Start Guide, Warranty Booklet " Product Description "Operating System: Wear OS by Google Sensors: Optical Heart Rate, Ambient Light, Barometric Altimeter, GPS, Accelerometer, Gyroscope, Microphone Processor: Qualcomm Snapdragon Wear 2100 Connectivity: Bluetooth(R) 4.1 Low Energy and Wi-Fi 802.11 b/g/n Storage: 4GB Water Resistant: Swimproof up to 30M, passes 10K stroke test What's in the Box: Smartwatch, USB Charging Cable, Quick Start Guide, Warranty Bookle

### Q3 (b): Write your code below

**1. The auto-grader will extract and use the following variables, DON'T change the their names:**

      self.topic_file  
      self.qrels_file  
      self.document_dir   
      self.file_list  
      self.index_sys  
      self.query_parser  
      self.searcher   



**2. DON'T change the names of the already defined funtions**  
**3. DON'T change the py_trec_eval function**  
**4. DON'T change the class names including CustomFilter, IRSystem, IRQ2, IRQ3, IRQ4**  
**5. DON'T change the CustomFilter class and DON'T create any new custom filter class that is used to define Whoosh schema**

In [16]:
class IRQ3(IRSystem):
    def create_index(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Please update self.index_sys which should have type whoosh.index.FileIndex
        """

        # DON't change the name of 'index_sys'

        # Analyzer taught in the lab
        analyzer = (
            RegexTokenizer() # Split text into individual word tokens
            | LowercaseFilter() # Make matching case-insensitive
            | StopFilter() # Remove common words such as "the" and "a"
            | StemFilter() # Reduce related words to a commoon root

        )

        # Store each document's path and create a searchable content field
        schema = Schema(file_path=ID(stored=True),
                        file_content=TEXT(analyzer=analyzer)
                        )

        # Create the Whoosh index in a temporary directory
        index_dir = tempfile.mkdtemp()
        self.index_sys = index.create_in(index_dir, schema)

    def add_files(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Add buffer to self.index_sys
        """

        # Open a buffered writer to add documents efficiently
        writer = writing.BufferedWriter (
            self.index_sys,
            period=None,
            limit=1000
        )

        try:
          # Read every product document and add it to the index
          for file_path in self.file_list:
            with open(file_path, encoding="utf-8") as file:
              file_content = file.read()

              writer.add_document (
                  file_path=file_path,
                  file_content=file_content
              )

        finally:
          # Close the writer and commit all indexed documents
          writer.close()

    def create_parser_searcher(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Please update self.query_parser and self.self.searcherwhich should have type whoosh.qparser.default.QueryParser and whoosh.searching.Searcher respectively
        """
         # DON't change the names of 'query_parser' and 'searcher'

         # OrGroup permits partial matching: a document can be retrieved without containing every query term. The scale rewards documents that match more of the query terms

        self.query_parser = QueryParser(
            "file_content",
            schema=self.index_sys.schema,
            group=qparser.OrGroup.factory(0.9)
        )

        # Create the searcher that ranks documents in the index
        self.searcher = self.index_sys.searcher()

    def perform_search(self, topic_phrase):
        """
        INPUT:
            topic_phrase: string
        OUTPUT:
            topicResults: whoosh.searching.Results

        NOTE: Utilize self.query_parser and self.searcher to calculate the result for topic_phrase
        """

        # Parse the query using partial OR matching
        partial_query = self.query_parser.parse(topic_phrase)

        # Remove any existing quotation marks before constructing an extract phrase version of the query
        cleaned_phrase = topic_phrase.replace('"', "")
        phrase_query = self.query_parser.parse(
            '"' + cleaned_phrase + '"'
        )

        # Retrieve partial matches while also rewarding documents in which the complete query phrase appears together
        combined_query = query.Or([
            partial_query,
            phrase_query
        ])

        # Return all matching documents in ranked order
        topicResults = self.searcher.search(
            combined_query,
            limit=None
        )

        return topicResults

In [17]:
q3 = IRQ3("product_search")

In [18]:
q3.add_files()

In [19]:
q3.py_trec_eval()

num_q                    100246  1.0000
num_ret                  100246  781.0000
num_rel                  100246  102.0000
num_rel_ret              100246  102.0000
map                      100246  0.9761
gm_map                   100246  -0.0242
Rprec                    100246  0.9118
bpref                    100246  0.9775
recip_rank               100246  1.0000
iprec_at_recall_0.00     100246  1.0000
iprec_at_recall_0.10     100246  1.0000
iprec_at_recall_0.20     100246  1.0000
iprec_at_recall_0.30     100246  1.0000
iprec_at_recall_0.40     100246  1.0000
iprec_at_recall_0.50     100246  0.9870
iprec_at_recall_0.60     100246  0.9870
iprec_at_recall_0.70     100246  0.9870
iprec_at_recall_0.80     100246  0.9659
iprec_at_recall_0.90     100246  0.9388
iprec_at_recall_1.00     100246  0.8095
P_5                      100246  1.0000
P_10                     100246  1.0000
P_15                     100246  1.0000
P_20                     100246  1.0000
P_30                     100246  

In [20]:
q3.print_rel_name('100246')

---------------------------Topic_id and Topic_phrase----------------------------------
100246 SPLOTY Tire Inflator Air Compressor Portable 12V
---------------------------Return documents----------------------------------
100246 Q0 1489405 0 107.834998 test
100246 Q0 1555783 1 48.626965 test
100246 Q0 1640998 2 40.054653 test
100246 Q0 121627 3 38.362932 test
100246 Q0 1503222 4 38.362932 test
100246 Q0 1294856 5 38.311082 test
100246 Q0 1657270 6 37.922988 test
100246 Q0 1555557 7 37.826403 test
100246 Q0 766020 8 37.287219 test
100246 Q0 934348 9 37.118655 test
100246 Q0 406252 10 36.990940 test
100246 Q0 1216577 11 36.586634 test
100246 Q0 339634 12 36.444574 test
100246 Q0 1148601 13 36.419820 test
100246 Q0 205050 14 36.126015 test
100246 Q0 407108 15 35.965623 test
100246 Q0 103372 16 35.828909 test
100246 Q0 861067 17 35.770133 test
100246 Q0 746471 18 35.728641 test
100246 Q0 627160 19 35.605081 test
100246 Q0 128561 20 35.567206 test
100246 Q0 1191987 21 35.549846 test
100246 Q

In [21]:
INDEX_Q3 = q3.index_sys
QP_Q3 = q3.query_parser
SEARCHER_Q3 = q3.searcher

### Q3 (c): Provide answer to Q3 (c) here [markdown cell]

I added lowercasing, stop-word removal, stemming, partial OR matching, and phrase matching. Overall P_10 increased from 0.2800 in Q2 to 0.8400 in Q3, and Q3 returned results for five queries that produced no results in Q2. For topic 100350, the false positive 1453918 moved from rank 6 to rank 31, while the false negative 1050182 improved from not being retrieved to rank 15.

In [22]:
results = q3.perform_search("Google Wi-Fi System Mesh")

ranks = {
    os.path.basename(result["file_path"]): rank
    for rank, result in enumerate(results)
}

print("False positive 1453918 rank:", ranks.get("1453918", "not retrieved"))
print("False negative 1050182 rank:", ranks.get("1050182", "not retrieved"))

False positive 1453918 rank: 31
False negative 1050182 rank: 15


### Q3 (d): Provide answer to Q3 (d) here [markdown cell]

Yes.

### Q3 (e): Provide answer to Q3 (e) here [markdown cell]

No. Some queries improved while others remained the same, but none had a lower P@10.

### Q3 (f): Provide answer to Q3 (f) here [markdown cell]

The modification was successful because partial matching and text normalization improved retrieval across the collection and allowed the system to return results for five previously unanswered queries. However, topic 100436 still had a P_10 of 0, so the system could be improved further.

## Question 4

### Q4a) Write your code below

**1. The auto-grader will extract and use the following variables, DON'T change the their names:**

      self.topic_file  
      self.qrels_file  
      self.document_dir
      self.text_embedding
      self.file_list  
      self.index_sys  
      self.query_parser  
      self.searcher   



**2. DON'T change the names of the already defined funtions**  
**3. DON'T change the py_trec_eval function**  
**4. DON'T change the class names including CustomFilter, IRSystem, IRQ2, IRQ3, IRQ4**  
**5. DON'T change the CustomFilter class and DON'T create any new custom filter class that is used to define Whoosh schema**

In [23]:
class IRQ4(IRSystem):
    def create_index(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Please update self.index_sys which should have type
            chromadb.collection.Collection with name `ir_assignment_[utorid]`
        """
        # Create a Chroma database client
        chroma_client = chromadb.Client()

        # Modify collection_name as `ir_assignment_[utorid]`
        name = 'ir-assignment-wangbet1'

        ## DON't Modify Following Line
        if name in [collection.name for collection in
                               chroma_client.list_collections()]:
            chroma_client.delete_collection(name)

        # Your Code Here, DON't change the name of 'index_sys'
        # Query and document embeddings will be compared using cosine distance
        self.index_sys = chroma_client.create_collection(
            name=name,
            metadata={"hnsw:space":"cosine"}
        )

    def add_files(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Add embedding of each document from self.text_embedding,
              and content of each document, to self.index_sys
        """

        # Go through every product document in the dataset
        # Doc_num is a counter, while file_path is the document's location

        for doc_num, file_path in enumerate(self.file_list):
            with open(file_path, encoding="utf-8") as file:
                file_content = file.read()

            # The filename is also the document ID
            document_id = os.path.basename(file_path)

            # Add three pieces of information to the Chroma collection:
            #1. the document ID,
            #2. the orginal document text,
            #3. the provided precomputed Qwen embedding
            self.index_sys.add(
                ids=[document_id],
                documents=[file_content],
                embeddings=[self.text_embedding[document_id]]
            )

    def create_parser_searcher(self):
        """
        INPUT:
            None
        OUTPUT:
            None

        NOTE: Please update self.query_parser,
              which should have type sentence_transformers.SentenceTransformer with Qwen/Qwen3-embedding-0.6b model


        """
        # DON't change the names of 'query_parser' and 'searcher'
        # Load the Qwen embedding model, converting a text query into a numerical vector representing its meaning
        self.query_parser = SentenceTransformer('Qwen/Qwen3-embedding-0.6b')


        # DON't modify the following line
        # Use the same Chroma collection as the searcher, containing the document embeddings that the query embedding will be compared against
        self.searcher = self.index_sys

    def perform_search(self, topic_phrase):
        """
        INPUT:
            topic_phrase: string
        OUTPUT:
            topicResults: dict

        NOTE: Utilize self.query_parser and self.searcher to calculate the result for topic_phrase
        """

        # Converting query text into an embedding using the Qwen model. Chroma expects the embedding as a regular Python list.
        query_embedding = self.query_parser.encode(
            topic_phrase,
            normalize_embeddings = False
        ).tolist()

        # Compare the query embedding with the stored document embeddings. Return up to 10 documents with the smallest cosine distances. Min () prevents requesting more documents than the collection contains.
        topicResults = self.searcher.query(
            query_embeddings = [query_embedding],
            n_results=min(10,self.searcher.count())
        )
        return topicResults

In [24]:
q4= IRQ4("product_search")
q4.add_files()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [25]:
q4.py_trec_eval()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

num_q                    100246  1.0000
num_ret                  100246  10.0000
num_rel                  100246  102.0000
num_rel_ret              100246  10.0000
map                      100246  0.0980
gm_map                   100246  -2.3224
Rprec                    100246  0.0980
bpref                    100246  0.0980
recip_rank               100246  1.0000
iprec_at_recall_0.00     100246  1.0000
iprec_at_recall_0.10     100246  0.0000
iprec_at_recall_0.20     100246  0.0000
iprec_at_recall_0.30     100246  0.0000
iprec_at_recall_0.40     100246  0.0000
iprec_at_recall_0.50     100246  0.0000
iprec_at_recall_0.60     100246  0.0000
iprec_at_recall_0.70     100246  0.0000
iprec_at_recall_0.80     100246  0.0000
iprec_at_recall_0.90     100246  0.0000
iprec_at_recall_1.00     100246  0.0000
P_5                      100246  1.0000
P_10                     100246  1.0000
P_15                     100246  0.6667
P_20                     100246  0.5000
P_30                     100246  0.

In [26]:
q4.print_rel_name('100246')

---------------------------Topic_id and Topic_phrase----------------------------------
100246 SPLOTY Tire Inflator Air Compressor Portable 12V
---------------------------Return documents----------------------------------
100246 Q0 1489405 0 0.072483 test
100246 Q0 1555783 1 0.085741 test
100246 Q0 766020 2 0.129460 test
100246 Q0 1107134 3 0.140477 test
100246 Q0 1569623 4 0.140477 test
100246 Q0 1552618 5 0.142278 test
100246 Q0 1148601 6 0.143078 test
100246 Q0 128561 7 0.145800 test
100246 Q0 853409 8 0.148182 test
100246 Q0 1657270 9 0.149713 test
---------------------------Relevant documents----------------------------------
100246 0 10037 1
100246 0 1030471 1
100246 0 103372 1
100246 0 1044 1
100246 0 1044436 1
100246 0 106086 1
100246 0 1079438 1
100246 0 1086239 1
100246 0 1093412 1
100246 0 10966 1
100246 0 1105375 1
100246 0 1107134 1
100246 0 1137791 1
100246 0 1140524 1
100246 0 1148601 1
100246 0 116655 1
100246 0 1176585 1
100246 0 117671 1
100246 0 1187066 1
100246 0 119

In [32]:
INDEX_Q4 = q4.index_sys
QP_Q4 = q4.query_parser
SEARCHER_Q4 = q4.searcher

### Q4 (b): Provide answer to Q4 (b) here [markdown cell]


No, not compared with the modified Whoosh system from Q3. Dense retrieval achieved an overall P_10 of 0.8333, which was slightly lower than Q3’s P_10 score of 0.8400. However, it outperformed the baseline Q2 system’s reported score of 0.2800.

### Q4 (c): Provide answer to Q4 (c) here [markdown cell]


Yes. Compared with Q3, topic 200303 improved from 0.1000 to 0.4000, while topic 100350 decreased from 1.0000 to 0.4000.

### Q4 (d): Provide answer to Q4 (d) here [markdown cell]

Although dense retrieval improved some queries by capturing semantic similarity, its overall P@10 was slightly lower than the improved Whoosh system. Performance might be enhanced by combining dense retrieval with keyword matching so that semantic similarity is used without losing exact product-name matches.

## Validation

In [27]:
# Run the following cells to make sure your code returns the correct value types

In [28]:
from whoosh.index import FileIndex
from whoosh.qparser import QueryParser
from whoosh.searching import Searcher
import os.path

### Q2 Validation

In [29]:
q2 = IRQ2("product_search")
assert(isinstance(q2.index_sys, FileIndex)), "Index Type"
assert(isinstance(q2.query_parser, QueryParser)), "Query Parser Type"
assert(isinstance(q2.searcher, Searcher)), "Searcher Type"
print("Q2 Types Validated")

Q2 Types Validated


### Q3 Validation

In [30]:
q3 = IRQ3("product_search")
assert(isinstance(q3.index_sys, FileIndex)), "Index Type"
assert(isinstance(q3.query_parser, QueryParser)), "Query Parser Type"
assert(isinstance(q3.searcher, Searcher)), "Searcher Type"
print("Q3 Types Validated")

Q3 Types Validated


### Q4 Validation

In [31]:
q4 = IRQ4("product_search")
assert(isinstance(q4.index_sys, chromadb.Collection)), "Collection Type"
assert(isinstance(q4.query_parser, SentenceTransformer)), " SentenceTransformer Type"
assert(isinstance(q4.searcher, chromadb.Collection)), "Collection Type"
print("Q4 Types Validated")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Q4 Types Validated
